# Movie_Feature (Supporting class)

In [2]:
import numpy as np
import scipy.sparse as sparse
from sklearn.decomposition import TruncatedSVD

class MovieFeatureExtractor:
    """
    Lớp trích xuất đặc trưng nội dung phim (Content-Based Filtering)
    sử dụng Latent Semantic Analysis (LSA) trên ma trận thưa TF-IDF.
    """
    def __init__(self, k_dimensions=100, random_state=42):
        """
        Khởi tạo Extractor.
        
        Parameters:
        - k_dimensions: Số lượng chủ đề ẩn (Latent topics) muốn giữ lại.
        - random_state: Seed để tái tạo lại kết quả.
        """
        self.k_dimensions = k_dimensions
        self.random_state = random_state
        self.variance_retained = 0.0
        
        # Khởi tạo mô hình TruncatedSVD
        self.svd_model = TruncatedSVD(
            n_components=self.k_dimensions, 
            algorithm='randomized', 
            random_state=self.random_state
        )

    def fit(self, tfidf_mat):
        """
        Huấn luyện mô hình để tìm ra không gian vector ẩn (Latent space).
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần để train.
        
        Returns:
        - self: Trả về chính object hiện tại.
        """
        print(f"[*] Đang huấn luyện mô hình SVD (Fit) với k = {self.k_dimensions}...")
        
        # Fit ma trận TF-IDF để học U, Sigma, V^T
        self.svd_model.fit(tfidf_mat)
        
        # Phân tích phương sai (Độ tin cậy của mô hình)
        self.variance_retained = np.sum(self.svd_model.explained_variance_ratio_) * 100
        
        print(f"[+] Huấn luyện hoàn tất!")
        print(f"[+] Tổng lượng thông tin được giữ lại (Explained Variance): {self.variance_retained:.2f}%")
        
        # Cảnh báo học thuật
        if self.variance_retained < 60:
            print("[!] Cảnh báo: Lượng thông tin giữ lại < 60%. Bạn nên cân nhắc tăng k_dimensions.")
            
        return self

    def transform(self, tfidf_mat):
        """
        Chiếu ma trận TF-IDF vào không gian ẩn đã được học từ bước fit.
        
        Parameters:
        - tfidf_mat: Ma trận scipy.sparse cần biến đổi.
        
        Returns:
        - item_content_profiles: Ma trận đặc trưng nội dung của video (Dense array)
        """
        print(f"[*] Đang biến đổi dữ liệu (Transform)...")
        
        # Transform dữ liệu
        item_content_profiles = self.svd_model.transform(tfidf_mat)
        
        print(f"[+] Kích thước Item Content Profiles: {item_content_profiles.shape}")
        return item_content_profiles

    def fit_transform(self, tfidf_mat):
        """
        Kết hợp fit và transform trong 1 bước (tối ưu hóa hiệu năng tính toán).
        """
        self.fit(tfidf_mat)
        return self.transform(tfidf_mat)



# Class build data for training and testing

In [3]:
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

class RecommenderDatasetBuilder:
    def __init__(self, ratings_npz_path, ratings_pkl_path, movies_npz_path, movies_pkl_path, k_dimensions=100, random_state=42):
        """
        Khởi tạo class và load toàn bộ dữ liệu từ các đường dẫn file.
        """
        print("Đang load dữ liệu...")
        
        # 1. Load file Pickle (.pkl)
        with open(ratings_pkl_path, 'rb') as f:
            self.collab_pkl_data = pickle.load(f)
            
        with open(movies_pkl_path, 'rb') as f:
            self.tfidf_pkl_data = pickle.load(f)

        # 2. Load file NPZ (.npz)
        # Ma trận ratings lưu dạng sparse (scipy)
        self.collab_mat = sp.load_npz(ratings_npz_path)

        # movies_npz_path là ma trận TF-IDF sparse, KHÔNG phải movie_factors_matrix đã nén bằng np.savez
        # Vì vậy cần chiếu qua SVD để tạo movie_factors_matrix đúng như MovieFeatureExtractor.
        self.tfidf_mat = sp.load_npz(movies_npz_path)

        # Đảm bảo k hợp lệ cho TruncatedSVD: 1 <= k < min(n_samples, n_features)
        max_k = min(self.tfidf_mat.shape) - 1
        if max_k < 1:
            raise ValueError(f"movies_npz_path có shape không hợp lệ cho SVD: {self.tfidf_mat.shape}")
        effective_k = min(k_dimensions, max_k)

        svd = TruncatedSVD(
            n_components=effective_k,
            algorithm='randomized',
            random_state=random_state,
        )
        self.movie_factors_matrix = svd.fit_transform(self.tfidf_mat)

        print(f"Shape của ma trận phim (movie factors): {self.movie_factors_matrix.shape}")

        # 3. Khởi tạo các dictionary mapping
        self.userid_to_index_collab = self.collab_pkl_data['user_mapping']
        self.index_to_userid_collab = {v: k for k, v in self.userid_to_index_collab.items()}

        self.movieid_to_index_collab = self.collab_pkl_data['movie_mapping']
        self.index_to_movieid_collab = {v: k for k, v in self.movieid_to_index_collab.items()}

        self.movieid_to_index_tfidf = self.tfidf_pkl_data['movie_mapping']
        self.index_to_movieid_tfidf = {v: k for k, v in self.movieid_to_index_tfidf.items()}
        
        # Trích xuất dữ liệu từ sparse matrix
        self.rows, self.cols = self.collab_mat.nonzero()
        self.values = self.collab_mat.data

    def _calculate_user_profiles(self):
        """
        BƯỚC 1: TÍNH VECTOR TRUNG BÌNH CHO TỪNG USER
        """
        print("Đang tính toán vector trung bình cho từng User...")
        user_movie_vectors = {}

        for user_idx_in_collab, movie_idx_in_collab in zip(self.rows, self.cols):
            user_id = self.index_to_userid_collab.get(user_idx_in_collab, "Unknown User")
            movie_id = self.index_to_movieid_collab.get(movie_idx_in_collab, "Unknown Movie")
            movie_idx_in_tfidf = self.movieid_to_index_tfidf.get(movie_id, None)

            if movie_idx_in_tfidf is None:
                continue  # Bỏ qua nếu không map được

            # Lấy vector phim và flatten
            movie_vector = self.movie_factors_matrix[movie_idx_in_tfidf].flatten()
            
            # Lưu vào danh sách các phim của user này
            if user_id not in user_movie_vectors:
                user_movie_vectors[user_id] = []
            user_movie_vectors[user_id].append(movie_vector)

        # Tính trung bình các vector phim cho mỗi user
        user_avg_profiles = {}
        for user_id, vectors in user_movie_vectors.items():
            user_avg_profiles[user_id] = np.mean(vectors, axis=0)
            
        return user_avg_profiles

    def _generate_dataset(self, user_avg_profiles):
        """
        BƯỚC 2: TẠO MA TRẬN X VÀ y CUỐI CÙNG
        """
        print("Đang tạo dataset (X, y)...")
        X_list = []
        y_list = []

        for user_idx_in_collab, movie_idx_in_collab, rating in zip(self.rows, self.cols, self.values):
            user_id = self.index_to_userid_collab.get(user_idx_in_collab, "Unknown User")
            movie_id = self.index_to_movieid_collab.get(movie_idx_in_collab, "Unknown Movie")
            movie_idx_in_tfidf = self.movieid_to_index_tfidf.get(movie_id, None)

            if movie_idx_in_tfidf is None:
                continue

            # 1. Lấy vector phim
            movie_vector = self.movie_factors_matrix[movie_idx_in_tfidf].flatten()
            
            # 2. Lấy vector user trung bình
            user_vector = user_avg_profiles[user_id]
            
            # 3. Nối (concatenate) hai vector
            combined_features = np.concatenate((user_vector, movie_vector))
            
            X_list.append(combined_features)
            y_list.append(rating)

        X = np.array(X_list)
        y = np.array(y_list)
        
        return X, y

    def build(self):
        """
        Hàm chính (Public method) để gọi và thực thi toàn bộ quy trình
        """
        user_avg_profiles = self._calculate_user_profiles()
        X, y = self._generate_dataset(user_avg_profiles)
        
        print("\n--- HOÀN TẤT ---")
        print(f"Số lượng mẫu đã tạo: {len(X)}")
        if len(X) > 0:
            print(f"Kích thước của ma trận X: {X.shape}")
            print(f"Kích thước của vector y: {y.shape}")
            # In ra một phần nhỏ để kiểm tra thay vì cả mảng to
            print(f"Mẫu X đầu tiên (5 phần tử đầu): {X[0][:5]} ...")
            
        return X, y

# Preparing Data for training

In [4]:
from pathlib import Path
from datetime import datetime
import gc
import json
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================================================
# Shared cache + helpers (run once before training cells)
# =========================================================

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data" / "processed").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục data/processed từ thư mục hiện tại.")

processed_dir = project_root / "data" / "processed"
split_cache_dir = project_root / "artifacts" / "split_cache"
split_cache_dir.mkdir(parents=True, exist_ok=True)


def regression_metrics(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def append_run_log(log_path, payload):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write("\n" + "=" * 90 + "\n")
        f.write(f"timestamp: {payload['timestamp']}\n")
        f.write(f"model: {payload['model_name']}\n")
        f.write(f"train_set: {payload['train_name']} | n_samples={payload['train_n']} | n_features={payload['n_features']}\n")
        f.write(f"tracked_test: {payload['tracked_name']} | n_samples={payload['tracked_n']}\n")
        f.write(f"untracked_test: {payload['untracked_name']} | n_samples={payload['untracked_n']}\n")
        f.write("tracked_metrics: " + json.dumps(payload["tracked_metrics"], ensure_ascii=False) + "\n")
        f.write(f"tracked_r2: {payload['tracked_metrics']['r2']:.6f}\n")
        f.write("untracked_metrics: " + json.dumps(payload["untracked_metrics"], ensure_ascii=False) + "\n")
        f.write(f"untracked_r2: {payload['untracked_metrics']['r2']:.6f}\n")
        f.write("extra: " + json.dumps(payload["extra"], ensure_ascii=False) + "\n")
        f.write("=" * 90 + "\n")


def split_cache_path(split_name):
    return split_cache_dir / f"{split_name}.npz"


def save_split_cache(split_name, builder):
    X_split, y_split = builder.build()
    np.savez_compressed(
        split_cache_path(split_name),
        X=np.asarray(X_split, dtype=np.float32),
        y=np.asarray(y_split, dtype=np.float32),
    )
    print(f"Saved {split_name} cache to: {split_cache_path(split_name)}")
    print(f"{split_name} cache shapes: X={X_split.shape}, y={y_split.shape}")
    del X_split, y_split
    gc.collect()


def prepare_split_cache(processed_dir, force=False):
    expected_files = {
        "train": split_cache_path("train"),
        "tracked_test": split_cache_path("tracked_test"),
        "untracked_test": split_cache_path("untracked_test"),
    }
    if not force and all(path.exists() for path in expected_files.values()):
        print(f"Split cache already exists in: {split_cache_dir}")
        return expected_files

    train_builder = RecommenderDatasetBuilder(
        ratings_npz_path=processed_dir / "train_warm_ratings_matrix.npz",
        ratings_pkl_path=processed_dir / "train_warm_ratings_mapping.pkl",
        movies_npz_path=processed_dir / "train_warm_movies_matrix.npz",
        movies_pkl_path=processed_dir / "train_warm_movies_mapping.pkl",
        k_dimensions=100,
        random_state=42,
    )
    save_split_cache("train", train_builder)

    tracked_builder = RecommenderDatasetBuilder(
        ratings_npz_path=processed_dir / "test_warm_ratings_matrix.npz",
        ratings_pkl_path=processed_dir / "test_warm_ratings_mapping.pkl",
        movies_npz_path=processed_dir / "train_warm_movies_matrix.npz",
        movies_pkl_path=processed_dir / "train_warm_movies_mapping.pkl",
        k_dimensions=100,
        random_state=42,
    )
    save_split_cache("tracked_test", tracked_builder)

    untracked_builder = RecommenderDatasetBuilder(
        ratings_npz_path=processed_dir / "cold_ratings_matrix.npz",
        ratings_pkl_path=processed_dir / "cold_ratings_mapping.pkl",
        movies_npz_path=processed_dir / "cold_movies_matrix.npz",
        movies_pkl_path=processed_dir / "cold_movies_mapping.pkl",
        k_dimensions=100,
        random_state=42,
    )
    save_split_cache("untracked_test", untracked_builder)

    return expected_files


def load_cached_split(split_name):
    cache_file = split_cache_path(split_name)
    with np.load(cache_file) as cached_data:
        X_split = cached_data["X"].astype(np.float32, copy=False)
        y_split = cached_data["y"].astype(np.float32, copy=False)
    return X_split, y_split


prepare_split_cache(processed_dir, force=False)

Đang load dữ liệu...
Shape của ma trận phim (movie factors): (5151, 100)
Đang tính toán vector trung bình cho từng User...
Đang tạo dataset (X, y)...

--- HOÀN TẤT ---
Số lượng mẫu đã tạo: 8582160
Kích thước của ma trận X: (8582160, 200)
Kích thước của vector y: (8582160,)
Mẫu X đầu tiên (5 phần tử đầu): [ 0.12404398 -0.00380396 -0.00836936  0.00813288 -0.01039773] ...
Saved train cache to: e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\artifacts\split_cache\train.npz
train cache shapes: X=(8582160, 200), y=(8582160,)
Đang load dữ liệu...
Shape của ma trận phim (movie factors): (5151, 100)
Đang tính toán vector trung bình cho từng User...
Đang tạo dataset (X, y)...

--- HOÀN TẤT ---
Số lượng mẫu đã tạo: 2145434
Kích thước của ma trận X: (2145434, 200)
Kích thước của vector y: (2145434,)
Mẫu X đầu tiên (5 phần tử đầu): [ 0.12283383  0.0044892  -0.01068475  0.00036507 -0.00925839] ...
Saved tracked_test cache to: e:\Year_3\Semester_2\Machine Learning\BTL\video_rec_system\arti

{'train': WindowsPath('e:/Year_3/Semester_2/Machine Learning/BTL/video_rec_system/artifacts/split_cache/train.npz'),
 'tracked_test': WindowsPath('e:/Year_3/Semester_2/Machine Learning/BTL/video_rec_system/artifacts/split_cache/tracked_test.npz'),
 'untracked_test': WindowsPath('e:/Year_3/Semester_2/Machine Learning/BTL/video_rec_system/artifacts/split_cache/untracked_test.npz')}

# LightGBM vs XGBoost

In [ ]:
import gc
import lightgbm as lgb
import xgboost as xgb

# =========================================================
# LightGBM + XGBoost evaluation from cached float32 splits
# =========================================================


def train_and_evaluate_lightgbm(X_train, y_train, X_tracked_test, y_tracked_test, log_dir):
    train_data = lgb.Dataset(X_train, label=y_train, free_raw_data=True)
    tracked_valid = lgb.Dataset(X_tracked_test, label=y_tracked_test, reference=train_data, free_raw_data=True)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "learning_rate": 0.01,
        "num_leaves": 63,
        "max_depth": -1,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "n_jobs": -1,
        "verbosity": -1,
    }

    model = lgb.train(
        params,
        train_data,
        num_boost_round=10000,
        valid_sets=[train_data, tracked_valid],
        valid_names=["train", "tracked_valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100),
        ],
    )

    best_iteration = getattr(model, "best_iteration", None)
    tracked_pred = model.predict(X_tracked_test, num_iteration=best_iteration)
    tracked_metrics = regression_metrics(y_tracked_test, tracked_pred)

    X_untracked_test, y_untracked_test = load_cached_split("untracked_test")
    untracked_pred = model.predict(X_untracked_test, num_iteration=best_iteration)
    untracked_metrics = regression_metrics(y_untracked_test, untracked_pred)

    payload = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model_name": "LightGBM Regression",
        "train_name": "train_warm_ratings",
        "tracked_name": "test_warm_ratings",
        "untracked_name": "cold_ratings",
        "train_n": int(len(y_train)),
        "tracked_n": int(len(y_tracked_test)),
        "untracked_n": int(len(y_untracked_test)),
        "n_features": int(X_train.shape[1]),
        "tracked_metrics": tracked_metrics,
        "untracked_metrics": untracked_metrics,
        "extra": {
            "best_iteration": int(best_iteration or 0),
            "best_score": getattr(model, "best_score", {}),
            "num_boost_round": 10000,
            "early_stopping_rounds": 50,
        },
    }

    append_run_log(log_dir / "lightgbm_regression.log", payload)
    del X_untracked_test, y_untracked_test
    gc.collect()
    return model, payload


def train_and_evaluate_xgboost(X_train, y_train, X_tracked_test, y_tracked_test, log_dir):
    train_matrix = xgb.DMatrix(X_train, label=y_train)
    tracked_valid = xgb.DMatrix(X_tracked_test, label=y_tracked_test)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "booster": "gbtree",
        "eta": 0.01,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "tree_method": "auto",
        "verbosity": 0,
    }

    evals_result = {}
    model = xgb.train(
        params,
        train_matrix,
        num_boost_round=10000,
        evals=[(train_matrix, "train"), (tracked_valid, "tracked_valid")],
        evals_result=evals_result,
        callbacks=[xgb.callback.EarlyStopping(rounds=50, save_best=True)],
        verbose_eval=100,
    )

    best_iteration = getattr(model, "best_iteration", None)
    if best_iteration is None:
        tracked_pred = model.predict(tracked_valid)
    else:
        tracked_pred = model.predict(tracked_valid, iteration_range=(0, best_iteration + 1))
    tracked_metrics = regression_metrics(y_tracked_test, tracked_pred)

    X_untracked_test, y_untracked_test = load_cached_split("untracked_test")
    untracked_valid = xgb.DMatrix(X_untracked_test, label=y_untracked_test)
    if best_iteration is None:
        untracked_pred = model.predict(untracked_valid)
    else:
        untracked_pred = model.predict(untracked_valid, iteration_range=(0, best_iteration + 1))
    untracked_metrics = regression_metrics(y_untracked_test, untracked_pred)

    payload = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model_name": "XGBoost Regression",
        "train_name": "train_warm_ratings",
        "tracked_name": "test_warm_ratings",
        "untracked_name": "cold_ratings",
        "train_n": int(len(y_train)),
        "tracked_n": int(len(y_tracked_test)),
        "untracked_n": int(len(y_untracked_test)),
        "n_features": int(X_train.shape[1]),
        "tracked_metrics": tracked_metrics,
        "untracked_metrics": untracked_metrics,
        "extra": {
            "best_iteration": int(best_iteration or 0),
            "evals_result": evals_result,
            "num_boost_round": 10000,
            "early_stopping_rounds": 200,
        },
    }

    append_run_log(log_dir / "xgboost_regression.log", payload)
    del X_untracked_test, y_untracked_test, untracked_valid
    gc.collect()
    return model, payload


# Prepare float32 cached splits once.
X_train, y_train = load_cached_split("train")
X_tracked_test, y_tracked_test = load_cached_split("tracked_test")
log_dir = project_root / "artifacts" / "logs"

lgbm_model, lgbm_run = train_and_evaluate_lightgbm(
    X_train, y_train,
    X_tracked_test, y_tracked_test,
    log_dir,
)

xgb_model, xgb_run = train_and_evaluate_xgboost(
    X_train, y_train,
    X_tracked_test, y_tracked_test,
    log_dir,
)

print("LightGBM tracked metrics:", lgbm_run["tracked_metrics"])
print("LightGBM untracked metrics:", lgbm_run["untracked_metrics"])
print("LightGBM best_iteration:", lgbm_run["extra"]["best_iteration"])
print("XGBoost tracked metrics:", xgb_run["tracked_metrics"])
print("XGBoost untracked metrics:", xgb_run["untracked_metrics"])
print("XGBoost best_iteration:", xgb_run["extra"]["best_iteration"])
print("XGBoost sample counts:", {
    "train_n": xgb_run["train_n"],
    "tracked_n": xgb_run["tracked_n"],
    "untracked_n": xgb_run["untracked_n"],
})
print(f"Logs written to: {log_dir}")

del X_train, y_train, X_tracked_test, y_tracked_test
gc.collect()

Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 0.865379	tracked_valid's rmse: 0.839463
[200]	train's rmse: 0.863235	tracked_valid's rmse: 0.840537
Early stopping, best iteration is:
[17]	train's rmse: 0.869407	tracked_valid's rmse: 0.838685
[0]	train-rmse:0.87082	tracked_valid-rmse:0.83876
[100]	train-rmse:0.86548	tracked_valid-rmse:0.83951
[200]	train-rmse:0.86353	tracked_valid-rmse:0.84060
[214]	train-rmse:0.86334	tracked_valid-rmse:0.84072
LightGBM tracked metrics: {'rmse': 0.8386846068242471, 'mae': 0.6440443309835758, 'r2': 0.00021235821086074136}
LightGBM untracked metrics: {'rmse': 0.7582663460001501, 'mae': 0.5755237857062162, 'r2': -0.00023269748565657444}
LightGBM best_iteration: 17
XGBoost tracked metrics: {'rmse': 0.8386941893495797, 'mae': 0.6440449953079224, 'r2': 0.0001894831657409668}
XGBoost untracked metrics: {'rmse': 0.7582544041174607, 'mae': 0.5755414962768555, 'r2': -0.00020122528076171875}
XGBoost best_iteration: 15
XGBoost samp

0

# Ridge

In [5]:
from sklearn.linear_model import Ridge
import gc

# =========================================================
# Ridge Regression from cached float32 splits
# =========================================================


def train_and_evaluate_ridge(X_train, y_train, X_tracked_test, y_tracked_test, log_dir):
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_train, y_train)

    tracked_pred = model.predict(X_tracked_test)
    tracked_metrics = regression_metrics(y_tracked_test, tracked_pred)

    X_untracked_test, y_untracked_test = load_cached_split("untracked_test")
    untracked_pred = model.predict(X_untracked_test)
    untracked_metrics = regression_metrics(y_untracked_test, untracked_pred)

    payload = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model_name": "Ridge Regression",
        "train_name": "train_warm_ratings",
        "tracked_name": "test_warm_ratings",
        "untracked_name": "cold_ratings",
        "train_n": int(len(y_train)),
        "tracked_n": int(len(y_tracked_test)),
        "untracked_n": int(len(y_untracked_test)),
        "n_features": int(X_train.shape[1]),
        "tracked_metrics": tracked_metrics,
        "untracked_metrics": untracked_metrics,
        "extra": {
            "alpha": float(model.alpha),
            "solver": model.solver,
            "max_iter": model.max_iter,
        },
    }

    append_run_log(log_dir / "ridge_regression.log", payload)
    del X_untracked_test, y_untracked_test
    gc.collect()
    return model, payload


X_train, y_train = load_cached_split("train")
X_tracked_test, y_tracked_test = load_cached_split("tracked_test")
log_dir = project_root / "artifacts" / "logs"

ridge_model, ridge_run = train_and_evaluate_ridge(
    X_train, y_train,
    X_tracked_test, y_tracked_test,
    log_dir,
)

print("Ridge tracked metrics:", ridge_run["tracked_metrics"])
print("Ridge untracked metrics:", ridge_run["untracked_metrics"])
print(f"Ridge log written to: {log_dir / 'ridge_regression.log'}")

del X_train, y_train, X_tracked_test, y_tracked_test
gc.collect()

Ridge tracked metrics: {'rmse': 0.856289161009361, 'mae': 0.6597284078598022, 'r2': -0.042200565338134766}
Ridge untracked metrics: {'rmse': 0.9537063764142387, 'mae': 0.7515289783477783, 'r2': -0.5822927951812744}
Ridge log written to: c:\DONT SKIP CLASSES\HK6\HOC MAY\ASSIGNMENT\Ridge\artifacts\logs\ridge_regression.log


0

# Linear Regression

In [6]:
from sklearn.linear_model import LinearRegression
import gc

# =========================================================
# Linear Regression from cached float32 splits
# =========================================================


X_train, y_train = load_cached_split("train")
X_tracked_test, y_tracked_test = load_cached_split("tracked_test")
X_untracked_test, y_untracked_test = load_cached_split("untracked_test")
log_dir = project_root / "artifacts" / "logs"

linear_model = LinearRegression(n_jobs=-1)
linear_model.fit(X_train, y_train)

tracked_pred = linear_model.predict(X_tracked_test)
untracked_pred = linear_model.predict(X_untracked_test)
tracked_metrics = regression_metrics(y_tracked_test, tracked_pred)
untracked_metrics = regression_metrics(y_untracked_test, untracked_pred)

linear_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model_name": "Linear Regression",
    "train_name": "train_warm_ratings",
    "tracked_name": "test_warm_ratings",
    "untracked_name": "cold_ratings",
    "train_n": int(len(y_train)),
    "tracked_n": int(len(y_tracked_test)),
    "untracked_n": int(len(y_untracked_test)),
    "n_features": int(X_train.shape[1]),
    "tracked_metrics": tracked_metrics,
    "untracked_metrics": untracked_metrics,
    "extra": {
        "fit_intercept": bool(linear_model.fit_intercept),
        "copy_X": bool(linear_model.copy_X),
        "n_jobs": linear_model.n_jobs,
    },
}

append_run_log(log_dir / "linear_regression.log", linear_payload)

print("Linear Regression tracked metrics:", linear_payload["tracked_metrics"])
print("Linear Regression untracked metrics:", linear_payload["untracked_metrics"])
print(f"Linear Regression log written to: {log_dir / 'linear_regression.log'}")

del X_train, y_train, X_tracked_test, y_tracked_test, X_untracked_test, y_untracked_test
gc.collect()

Linear Regression tracked metrics: {'rmse': 0.8563492306757786, 'mae': 0.6597805023193359, 'r2': -0.042346835136413574}
Linear Regression untracked metrics: {'rmse': 0.9543830511007109, 'mae': 0.7520723938941956, 'r2': -0.5845389366149902}
Linear Regression log written to: c:\DONT SKIP CLASSES\HK6\HOC MAY\ASSIGNMENT\Ridge\artifacts\logs\linear_regression.log


66